In [1]:
# Add autoreload at the top of the notebook you're working on 
# in order for it to auto refresh when you change the 'project_package'
%load_ext autoreload
%autoreload 2

# Table of contents

[1. Import and align datasets for model training](#section-1)   
&emsp; [a. Load the processed recipe dataset](#section-1a)  
&emsp; [b. Load user rating data](#section-1b)  
&emsp; [c. Create user preference data](#section-1c)

[2. Initialize Chroma vectorstore with document embedding](#section-2)   
&emsp; [a. Create a collection for item data](#section-2a)  
&emsp; [b. Create a collection for user preferences](#section-2b)

[3. Train recommendation models ](#section-3)   
&emsp; [a. Preparing the training/test datasets and metric format](#section-3a)  
&emsp; [b. Cross validation all models for comparison](#section-3b)<br>
&emsp; [c. Retraining chosen models on full dataset](#section-3c)

[4. UI implementation](#section-4)   
&emsp; [4.1. Candidate selection](#section-4.1)  
&emsp; [4.1. Recommendation filtering  & Reranking](#section-4.2)<br>
&emsp;&emsp; [4.2.1. Item-content filtering](#section-4.2.1)<br>
&emsp;&emsp; [4.2.2. Collaboration filtering](#section-4.2.2)<br>
&emsp;&emsp; [4.2.3. Hybrid filtering](#section-4.2.3)



Import support libraries & modules

In [ ]:
import os,io
import warnings
import threadpoolctl
from pathlib import Path
import logging
import ast
logging.getLogger("httpx").setLevel(logging.WARNING)  # hide logging in cell when using chroma vectorstore
from dotenv import load_dotenv

import pandas as pd
import numpy as np
from langchain_community.document_compressors import FlashrankRerank
from flashrank import Ranker
from rank_bm25 import BM25Plus
from rectools.model_selection import cross_validate
from rectools.model_selection.random_split import RandomSplitter
from rectools.model_selection.last_n_split import LastNSplitter
from rectools.models import load_model,ImplicitItemKNNWrapperModel,ImplicitALSWrapperModel,LightFMWrapperModel,ImplicitBPRWrapperModel
from rectools.models.pure_svd import PureSVDModel
import implicit
from implicit.nearest_neighbours import TFIDFRecommender, BM25Recommender
from implicit.als import AlternatingLeastSquares as CPU_AlternatingLeastSquares
from implicit.gpu.als import AlternatingLeastSquares as GPU_AlternatingLeastSquares
from implicit.bpr import BayesianPersonalizedRanking as CPU_BayesianPersonalizedRanking
from implicit.gpu.bpr import BayesianPersonalizedRanking as GPU_BayesianPersonalizedRanking
from lightfm import LightFM

from project_package.data_collection.utility import load_chunks
from project_package.data_preprocessing.utils import create_user_preference,chroma_filter_operator
from project_package.modeling.recommendation_utils import (
    preprocessing_docs,generate_metric_objs,construct_rec_train_dataset,get_embedding_model,
    VectorstoreLoader,load_vector_store,generate_feature_constraint
    )
from project_package.data_preprocessing.default import (
    USER,ITEM,RATING_COL,RECIPE_TO_USE,
    RECIPE_COLUMN_MAPPING,REVIEW_COLUMN_MAPPING,REVIEW_TO_USE,
    DOC_TEMPLATE,USER_PROFILE_TEMPLATE,RECIPE_COLS,RECIPE_META_COLS,PREFERENCE_COLS
    )
from project_package.aws.data_access import pandas_sql_df,build_s3_client
from project_package.aws.model_store import upload_model_file,get_object_bytes

load_dotenv()  # load env variables from .evn
root_directory = Path(os.getcwd()).parent  #NOTE: update of notebook location changed


g:\Python\envs\capstone_test3\Lib\site-packages\lightfm\_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(
g:\Python\envs\capstone_test3\Lib\site-packages\dask\dataframe\__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


<a id='section-1'></a>
## 1. Import and align datasets for model training


a. Load the processed recipe dataset
<a id='section-1a'></a>

In [ ]:
#NOTE: might need update to retrieve the processed df from S3

recipe_df = pd.read_csv(root_directory / "data/processed/Recipes/recipes_chunk_0.csv",usecols=RECIPE_TO_USE)
# recipe_df = load_chunks(root_directory / "data/processed/Recipes","recipes_chunk_*.csv",usecols=to_use_columns)  # uncomment for full dataset
recipe_df.rename(columns = RECIPE_COLUMN_MAPPING,inplace=True)

recipe_df['ingredients'] = recipe_df['ingredients'].apply(ast.literal_eval)  # convert ingredient to list

recipe_category_df = pandas_sql_df("SELECT * from RECIPES")
recipe_category_df['original_id'] = recipe_category_df['original_id'].astype(int)
recipe_category_df['cooking_method'] = recipe_category_df['cooking_method'].str.strip("{}").str.split(",")

recipe_df = recipe_df.merge(recipe_category_df,on=['original_id','source'],how='inner')
recipe_df = recipe_df.dropna(subset = RECIPE_META_COLS)
recipe_df = recipe_df[recipe_df["ingredients"].apply(lambda x: x != [])]  # remove empty ingredient record

#Drop recipe that take too long, or calories value that are unreasonably high for a common meal
recipe_df = recipe_df.loc[(recipe_df['total_time']<=1440)&(recipe_df['calories']<= 5000)].reset_index(drop=True) 

recipe_df.head(2)

,original_id,recipe_name,instructions,calories,source,prep_time,cook_time,total_time,ingredients,who_score,...,recipe_id,cuisine,cooking_method,difficulty,protein_content,fiber_content,fat_content,carbohydrate_content,sodium_content,s3_key
0,39,Biryani,['Soak saffron in warm milk for 5 minutes and ...,1110.7,foodcom,240,25.0,265,"[saffron, milk, green chili, onion, garlic, ga...",1,...,2,asian,"[braise, simmer, fry, bake]",advanced,high,high,high,high,medium,recipes/2.json
1,40,Best Lemonade,"['Into a 1 quart Jar with tight fitting lid, p...",311.1,foodcom,30,5.0,35,"[sugar, lemon zest, water, lemon juice]",3,...,1,unknown,[unknown],intermediate,low,low,low,high,low,recipes/1.json


Retrieve all unique labels of recipe features to be used as dropdown list in the UI

In [52]:
if not os.path.exists(root_directory / 'data/processed/ui_feature_constraints.csv'):
    constraint_df = generate_feature_constraint(
        recipe_df,
        root_directory / "data/processed/ingredient_counts_ingredients_canonical_final_le_5_replace.csv"
    )
    constraint_df.to_csv(root_directory / 'data/processed/ui_feature_constraints.csv',index=False)
else:
    constraint_df = pd.read_csv(root_directory / 'data/processed/ui_feature_constraints.csv')
constraint_df.head(2)

,feature,value,type
0,calories,"[0.0, 4997.8]",numeric
1,prep_time,"[0, 1200]",numeric


b. Load user rating data
<a id='section-1b'></a>

In [ ]:
# should sort reviews by user first

#NOTE: might need update to retrieve the processed df from S3, remove any users that have less than n reviews
user_reviews = pd.read_csv(root_directory / "data/processed/Reviews/reviews_chunk_0.csv",usecols=REVIEW_TO_USE)
# recipe_df = load_chunks(root_directory / "data/processed/Reviews","reviews_chunk_*.csv",usecols=to_use_columns)  # uncomment for full dataset
user_reviews.rename(columns = REVIEW_COLUMN_MAPPING,inplace=True)

# Convert datetime data
user_reviews['modified_time'] = pd.to_datetime(user_reviews['modified_time'], utc=True)
user_reviews['modified_time'] = user_reviews['modified_time'].dt.tz_localize(None)

# merge with recipe_df to retrieve the unique recipe_id from Postgres database
user_reviews = user_reviews.merge(recipe_df[['original_id','source','recipe_id']],on=['original_id','source'],how='inner')
user_reviews = user_reviews.sort_values(['source','original_user_id']).reset_index(drop=True)

# Convert to user from difference source to unique IDs
user_reviews["user_id"] = user_reviews["original_user_id"].astype(str) + "_" + user_reviews["source"]

groupby = user_reviews.groupby('user_id')['rating'].count()
drop_index = groupby.index[groupby<10]  #NOTE: threshold for to keep user with this minimum number of reviews

user_reviews = user_reviews.loc[~(user_reviews['user_id'].isin(drop_index))].reset_index(drop=True)
user_reviews["user_id"], _ = pd.factorize(user_reviews["user_id"])

user_reviews = user_reviews[['user_id','recipe_id','rating','modified_time']]
user_reviews.head(3)

,user_id,recipe_id,rating,modified_time
0,0,14063,4,2002-02-19 12:32:18
1,0,17605,5,2005-02-05 15:06:54
2,0,7593,5,2002-05-02 14:20:30


c. Create user preference data
<a id='section-1c'></a>

In [6]:
# Generate user preference data

if not os.path.exists(root_directory / 'data/processed/user_preferences.csv'):
    #NOTE: We can update the preference threshold if we want to
    criteria_dict = dict(
        ingredients = (0.3,'multiple'),
        cuisine = (0.25,'single'), 
        cooking_method = (0.25,'multiple'),  
        difficulty = (0.35,'single'), 
        protein_content = (0.35,'single'), 
        fiber_content = (0.35,'single'), 
        fat_content = (0.35,'single'), 
        carbohydrate_content = (0.35,'single'),
        sodium_content = (0.35,'single')
    )

    preference_df = create_user_preference(
        recipe_df,ITEM,
        user_reviews,USER,
        criteria_dict,
        RATING_COL,
        user_batch=5000  #NOTE: Update batch for more speed
    )
    preference_df.to_csv(root_directory / 'data/processed/user_preferences.csv',index =False)
else:
    preference_df = pd.read_csv(root_directory / 'data/processed/user_preferences.csv')

preference_df.head(3)

,user_id,ingredients,cuisine,cooking_method,difficulty,protein_content,fiber_content,fat_content,carbohydrate_content,sodium_content
0,0,"['butter', 'salt']",['american'],['bake'],['intermediate'],['low'],['low'],[],[],['medium']
1,1,"['butter', 'egg', 'salt', 'sugar']",['american'],['bake'],['intermediate'],['low'],['low'],[],['high'],['medium']
2,2,[],['american'],[],['intermediate'],[],['low'],[],[],[]


## 2. Initialize Chroma vectorstore with document embedding
<a id='section-2'></a>

Load the embedding model

In [ ]:
embedding_model = get_embedding_model(
    huggingface_model_path="BAAI/bge-small-en-v1.5",  # NOTE: Change this embedding to foodbert later if necessary
    local_model_name="bge-small",
    device="cuda"
)
chroma_path = root_directory / "data/processed/chroma_db"  #NOTE: Change the Path if necessary

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

a. Create a collection for item data
<a id='section-2a'></a>

In [ ]:
#NOTE: Run the cell again when a batch is failed to continue

store_document = False  # Set to True when you need to run the embedding loading again
vector_loader = None

if store_document:
    if vector_loader is None:
        vector_loader = VectorstoreLoader(
            collection_name="recipe_collection",  #NOTE: Change the collection name if necessary,
            embedding = embedding_model,
            doc_template = DOC_TEMPLATE,
            input_data = recipe_df,
            format_cols = RECIPE_COLS,
            meta_cols = RECIPE_META_COLS, # include item ID for later filter tasks
            persist_directory = chroma_path,
            docID_col = ITEM
        )

    load_finished =  vector_loader.add_initial_documents(batch_size=1000)
    if load_finished:
        vectorstore = vector_loader.return_vectorstore()
        del vector_loader  # to reduce memory usage
else:
    vectorstore = load_vector_store(
        collection_name="recipe_collection",
        embedding_model=embedding_model,
        persist_directory=chroma_path
    )

Couldn't load the cloud Chroma vectorstore, switching to local storage.
Starting ingestion from index 0...


100%|██████████| 50/50 [10:03<00:00, 12.08s/it]

Calculation time for embeddings: 604.0s
Data ingestion completed.


b. Create a collection for user preferences
<a id='section-2b'></a>

In [ ]:
#NOTE: Run the cell again when a batch is failed to continue

store_user_pref = False  # Set to True when you need to run the embedding loading again
vector_loader = None

if store_user_pref:
    if vector_loader is None:
        vector_loader = VectorstoreLoader(
            collection_name="user_recipe_preference",  #NOTE: Change the collection name if necessary,
            embedding = embedding_model,
            doc_template = USER_PROFILE_TEMPLATE,
            input_data = preference_df,
            format_cols = PREFERENCE_COLS,
            meta_cols = None,
            persist_directory = chroma_path,
            docID_col = USER
        )

    load_finished =  vector_loader.add_initial_documents(batch_size=1000)
    if load_finished:
        user_vectorstore = vector_loader.return_vectorstore()
        del vector_loader  # to reduce memory usage
else:
    user_vectorstore = load_vector_store(
        collection_name="user_recipe_preference",
        embedding_model=embedding_model,
        persist_directory=chroma_path
    )

Couldn't load the cloud Chroma vectorstore, switching to local storage.
Starting ingestion from index 0...


100%|██████████| 2/2 [00:09<00:00,  4.95s/it]

Calculation time for embeddings: 9.9s
Data ingestion completed.


## 3. Train recommendation models 
<a id='section-3'></a>

a. Preparing the training/test datasets and metric format
<a id='section-3a'></a>

In [10]:
k = 10  # number of recommendation to create

# load data to Rectools format
dataset = construct_rec_train_dataset(
    user_reviews,
    recipe_df,
    preference_df,
    use_datetime = True
)

In [11]:
# Retrieve item embeddings from Chromastore
# we are using item embedding instead of onehot coded to compare item similarity
ids = vectorstore._collection.get(include=['embeddings'])['ids']  
doc_embeddings = vectorstore._collection.get(include=['embeddings'])['embeddings']
idx = pd.Index(ids,name='item_id',dtype=int)
embedding_docs = pd.DataFrame(doc_embeddings,index=idx)

# create metric objects for k recommendations
metrics = generate_metric_objs(embedding_docs,k=k)

### List of chosen recommendation models to train

* **ItemKNN model**

This is a wrapper model for item-item nearest neighbour models. Those models are item-content recommendation models, which based purely on the content and doesn't require user-item interaction or user preferences. A few recommendation model belongs to this type are BM25, TFIDF,etc.

* **SVD model**

This is a basic collaboration model where we only take the user-item score interactions then apply a dimension reduction algorithm to create embedding representation in a latent vector space. This allows the model to generate rating for unseen items and create recommendation to the users.

* **AlternatingLeastSquares**

Goal of the model is to present interactions matrix as a product of user(X) and item(Y) embeddings. Implicit ALS model treats all non-zero entries in the matrix as value. The actual weight of the interactions is treated as confidence in the observation. Zero entries receive low confidence since this they are treated as missing values and might actually hide items highly relevant to users. Non-zero entries with high confidence will have greater impact on the loss when not predicted correctly.

* **BayesianPersonalizedRanking**

Bayesian personalized ranking introduces a pairwise loss instead. For each user model takes a pair of items: one positive and one negative where positive item was present in user interactions and negative item wasn’t. The goal of the algorithm is to rank positive item higher then negative one. It is useful for cases when only positive interactions are present in data and when the goal is to maximize ROC AUC.

* **LightFM**

A hybrid latent representation recommender model.

The model learns embeddings (latent representations in a high-dimensional space) for users and items in a way that encodes user preferences over items. When multiplied together, these representations produce scores for every item for a given user; items scored highly are more likely to be interesting to the user.

The user and item representations are expressed in terms of representations of their features: an embedding is estimated for every feature, and these features are then summed together to arrive at representations for users and items


b. Cross validation all models for comparison
<a id='section-3b'></a>

<ins>skip this part if you already train the models</ins>

In [12]:
# For implicit ALS
os.environ["OPENBLAS_NUM_THREADS"] = "1"
threadpoolctl.threadpool_limits(1, "blas")

tfidf_model = ImplicitItemKNNWrapperModel(TFIDFRecommender())
bm25_model = ImplicitItemKNNWrapperModel(BM25Recommender(K1=1.5))
svd_model = PureSVDModel(factors=150,use_gpu=True)

if implicit.gpu.HAS_CUDA:
    als_model = ImplicitALSWrapperModel(GPU_AlternatingLeastSquares(factors=150,random_state=0))
    bpr_model = ImplicitBPRWrapperModel(GPU_BayesianPersonalizedRanking(factors=150,random_state=0))
else:
    als_model = ImplicitALSWrapperModel(CPU_AlternatingLeastSquares(factors=150,random_state=0))
    bpr_model = ImplicitBPRWrapperModel(CPU_BayesianPersonalizedRanking(factors=150,random_state=0))

#NOTE: Update the loss function when computer support thread
lightfm_model = LightFMWrapperModel(LightFM(no_components=100, loss="logistic",random_state=0))  

models = {
    "TFIDFRecommender":tfidf_model,
    "BM25Recommender":bm25_model,
    "PureSVDModel":svd_model,
    "AlternatingLeastSquares":als_model,
    "BayesianPersonalizedRanking":bpr_model,
    "LightFM":lightfm_model
}

# splitter = RandomSplitter(test_fold_frac=0.195, random_state=0,n_splits=5)  # test_fold_frac * n_splits can only be close to 100%, otherwise it's impossible to split
splitter = LastNSplitter(n=3,n_splits=5)

g:\Python\envs\capstone_test3\Lib\site-packages\rectools\models\pure_svd.py:113: UserWarning: Forced to use CPU. CuPy is not available.
  warnings.warn("Forced to use CPU. CuPy is not available.")


In [13]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    cv_results = cross_validate(
        dataset=dataset,
        splitter=splitter,
        models=models,
        metrics=metrics,
        k=k,
        filter_viewed=True,
    )

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
cross_validate_df = (
    pd.DataFrame(cv_results["metrics"])
    .drop(columns="i_split")
    .groupby(["model"], sort=False)
    .agg(["mean"])
)
cross_validate_df.columns = cross_validate_df.columns.droplevel(1)
cross_validate_df.to_csv(root_directory / 'models/recommendation_model_comparison.csv',index =False)
cross_validate_df

,Recall@10,Precision@10,NDCG@10,Novelty@10,AvgRecPopularity@10,Diversity@10,Serendipity@10
model,,,,,,,
TFIDFRecommender,0.002375,0.002375,0.000649,10.575631,0.000021,0.253424,8.693555e-07
BM25Recommender,0.003995,0.003995,0.001282,9.747291,0.000045,0.254088,1.789145e-06
PureSVDModel,0.012858,0.012858,0.004231,6.097334,0.000551,0.253182,2.337879e-06
AlternatingLeastSquares,0.000324,0.000324,0.000082,9.866712,0.000039,0.210596,2.030498e-07
BayesianPersonalizedRanking,0.007176,0.007176,0.002142,7.166348,0.000269,0.252401,2.192569e-06
LightFM,0.000096,0.000096,0.000040,10.610119,0.000005,0.252920,6.451976e-08


c. Retraining chosen models on full dataset
<a id='section-3c'></a>

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    svd_model = PureSVDModel(factors=150,use_gpu=True)

    if implicit.gpu.HAS_CUDA:
        als_model = ImplicitALSWrapperModel(GPU_AlternatingLeastSquares(factors=150,random_state=0))
    else:
        als_model = ImplicitALSWrapperModel(CPU_AlternatingLeastSquares(factors=150,random_state=0))
    #NOTE: Update the loss function when computer support thread
    lightfm_model = LightFMWrapperModel(LightFM(no_components=100, loss="logistic",random_state=0))  

    svd_model.fit(dataset)
    svd_model.save(root_directory / "models/recommendation_models/svd_recommendation_model.pkl")  #NOTE: Update Path if necessary

    als_model.fit(dataset)
    als_model.save(root_directory / "models/recommendation_models/als_recommendation_model.pkl")  #NOTE: Update Path if necessary

    lightfm_model.fit(dataset)
    lightfm_model.save(root_directory / "models/recommendation_models/lightFM_recommendation_model.pkl")  #NOTE: Update Path if necessary

    #NOTE: We can switch to save model in S3 bucket instead
    # s3_client = build_s3_client(
    #     os.environ['AWS_ACCESS_KEY'],
    #     os.environ['AWS_SECRET_KEY'],
    #     os.environ['AWS_SESSION_TOKEN']
    # )
    # # upload object from path
    # _ = upload_model_file(s3_client,root_directory / "models/recommendation_models/svd_recommendation_model.pkl",'svd_recommendation_model')
    # _ = upload_model_file(s3_client,root_directory / "models/recommendation_models/als_recommendation_model.pkl",'als_recommendation_model')
    # _ = upload_model_file(s3_client,root_directory / "models/recommendation_models/lightFM_recommendation_model.pkl",'lightfm_recommendation_model')

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

<a id='section-4'></a>
## 4. UI implementation


Load the trained recommendation models

In [ ]:
#NOTE: Load the models, for big model, we might need to load this from S3 or google drive
svd_model = load_model(root_directory / "models/recommendation_models/svd_recommendation_model.pkl")
als_model = load_model(root_directory / "models/recommendation_models/als_recommendation_model.pkl")
lightfm_model = load_model(root_directory / "models/recommendation_models/lightFM_recommendation_model.pkl")

# can LOAD model from S3 instead if it's there
# s3_client = build_s3_client(
#     os.environ['AWS_ACCESS_KEY'],
#     os.environ['AWS_SECRET_KEY'],
#     os.environ['AWS_SESSION_TOKEN']
# )
# svd_model = load_model(io.BytesIO(get_object_bytes(s3_client,'svd_recommendation_model')))
# als_model = load_model(io.BytesIO(get_object_bytes(s3_client,'als_recommendation_model')))
# lightfm_model = load_model(io.BytesIO(get_object_bytes(s3_client,'lightfm_recommendation_model')))

### 4.1. Candidate selection
<a id='section-4.1'></a>

In [17]:
# create dummy filter component values to mimic UI filters

test_dict = pd.DataFrame(dict(
    filter_type = ["filter_checklist","filter_dropdown","filter_slider","filter_slider"],
    filter_name = ['ingredients','cuisine','total_time','calories'],
    filter_value = [['egg', 'flour', 'butter','salt','sugar','wheat'],
                    ['fusion','american','mediterranean'],
                    [0,120],[0,4000]
                    ],
    priority_type = ['exact','exact','exact','exact']
))

operator_type_mapping = dict(
    filter_name = ["ingredients","cuisine","total_time","calories"],
    # record_type = ['list','string','number','number'],
    operator_type = ['$or:$contains','$in','$range','$range']
)

# map UI feature names to database name if they are different
name_mapping = {
}


In [18]:
filter_operators = chroma_filter_operator(test_dict,operator_type_mapping,name_mapping)

retriever = vectorstore.as_retriever(
    search_kwargs = dict(
        k=1000,  # We retrieve the best 1000 results
        filter=filter_operators
    )
)

test_query = "a meat hamburger"

retrieved_items =  retriever.invoke(test_query)  # retrieve the top search result from the vectorstore that match the query and metadata filter conditions

candidate_ids = [item.id for item in retrieved_items]

result_docs = [item.page_content for item in retrieved_items]

retrieved_items[:5]

[Document(id='5114', metadata={'cuisine': 'american', 'calories': 1708.2, 'difficulty': 'intermediate', 'fsa_score': 23, 'carbohydrate_content': 'medium', 'ingredients': ['beef', 'onion', 'water', 'evaporated milk', 'salt', 'worcestershire sauce', 'pepper', 'horseradish', 'mustard', 'chive', 'blue cheese', 'sesame seed', 'black olive', 'dill pickle'], 'prep_time': 10, 'protein_content': 'medium', 'sodium_content': 'high', 'recipe_id': 5114, 'fiber_content': 'medium', 'who_score': 0, 'cooking_method': ['broil', 'grill'], 'fat_content': 'high', 'cook_time': 15.0, 'total_time': 25}, page_content="\nThe recipe name:Basic Hamburgers.\nRecipe instruction:\n['Mix all ingredients (including those in the variations that you choose) together.', 'Shape mixture into 6 patties, each about 3/4-inch thick.', 'Broil or grill patties 4-inches from the heat, turning once, to desired doneness, 10 to 15 minutes.', 'Nice served on toasted buns with favorite toppings.']\nIngredient list:\n['beef', 'onion', 

### 4.2. Recommendation filtering  & Reranking
<a id='section-4.2'></a>

This step is used to further choosing the top recommendations that match the user preferences and query content before re-ranking

#### 4.2.1. Item-content filtering
<a id='section-4.2.1'></a>

We can't use the BM25Recommender model in Rectools for production, because it still need user ratings, but in the UI, we don't actually have user ratings for new users (cold start), so we need to build a pipeline that doesn't utilize user ratings to recommend.

We will be using the BM25Plus model for matching item-contents between the query and the documents

In [19]:
tokenized_corpus = preprocessing_docs(result_docs)
tokenized_query = preprocessing_docs(test_query)

#BM vectorizer model
bm25 = BM25Plus(tokenized_corpus)

doc_scores = bm25.get_scores(tokenized_query)

top_100 = np.array(candidate_ids)[np.argsort(doc_scores)[::-1][:100]]  # sort the similarity score descendingly
top_100_docs = vectorstore.get_by_ids(top_100)
top_100_docs[:5]

[Document(id='890', metadata={'prep_time': 20, 'sodium_content': 'high', 'difficulty': 'intermediate', 'fiber_content': 'high', 'protein_content': 'high', 'ingredients': ['hamburger', 'bacon', 'rice', 'tomato', 'salt', 'pea', 'cheese'], 'who_score': 0, 'fsa_score': 22, 'cuisine': 'american', 'recipe_id': 890, 'cooking_method': ['bake'], 'cook_time': 60.0, 'calories': 899.9, 'carbohydrate_content': 'high', 'total_time': 80, 'fat_content': 'high'}, page_content="\nThe recipe name:Hamburger Casserole.\nRecipe instruction:\n['Cook all meat until browned; drain fat.', 'Place remaining ingredients in alternate  layers in a casserole and top with bread crumbs.', 'Drizzle with 2 teaspoons melted  butter and bake 1 hour at 350°F.']\nIngredient list:\n['hamburger', 'bacon', 'rice', 'tomato', 'salt', 'pea', 'cheese']\nCuisine:american\nCooking method:['bake']\nThe difficulty is intermediate.\nProtein content is high.\nFiber content is high.\nFat content is high.\nCarbohydrate content is high.\nSo

Rerank the documents

In [ ]:
# Need to remapping the document ID because using FlashrankRerank the document id in the result will be overwriten with index from 0->n
doc_ids_map = dict(
    zip(range(len(top_100)),top_100.tolist())
)

reranker = Ranker(
    model_name="ms-marco-MiniLM-L-12-v2",  # NOTE: Can change to a different Flashrank model of your liking
    cache_dir=os.environ["FLASHRANK_PATH"]
)
compressor = FlashrankRerank(client=reranker,top_n=k)

In [21]:
rerank_result = compressor.compress_documents(
    top_100_docs,
    query = test_query + "and lot's of vegetable "  #NOTE: can add extra context here to the query
)

for doc in rerank_result:  # reranking override the doc id so need to change it bank
    doc.metadata['id'] = doc_ids_map[doc.metadata['id']]

data = []

for doc in rerank_result:   # your list of Document objects
    row = doc.metadata.copy()
    row["page_content"] = doc.page_content
    data.append(row)

pd.DataFrame(data)


,id,relevance_score,who_score,cuisine,prep_time,fat_content,ingredients,calories,carbohydrate_content,cooking_method,fsa_score,fiber_content,protein_content,difficulty,cook_time,sodium_content,recipe_id,total_time,page_content
0,37971,0.390067,1,american,20,low,"[hamburger, onion, beef broth, carrot, celery,...",174.4,low,"[sautee, simmer]",3,low,medium,intermediate,45.0,high,23861,65,\nThe recipe name:Hamburger Vegetable Soup.\nR...
1,5126,0.355797,0,american,30,high,"[beef, onion, garlic powder, garlic, salt, pep...",416.4,medium,"[fry, bake]",15,low,medium,intermediate,40.0,high,43211,70,\nThe recipe name:Hamburger Pacific.\nRecipe i...
2,23861,0.276528,1,american,15,medium,"[beef, salt, pepper, ketchup, garlic powder, p...",239.1,low,"[simmer, grill]",7,low,medium,intermediate,30.0,high,29388,45,\nThe recipe name:Hamburgers with Brown Gravy ...
3,39729,0.124354,3,american,30,medium,"[hamburger, green bean, onion, sharp cheddar c...",418.3,high,"[bake, simmer]",5,high,medium,intermediate,35.0,high,24915,65,\nThe recipe name:Hamburger Casserole.\nRecipe...
4,19549,0.074346,3,american,10,low,"[hamburger, onion, potato, chicken bouillon cu...",355.8,high,"[boil, fry, simmer]",-2,high,medium,intermediate,25.0,medium,55686,35,\nThe recipe name:Hamburger Fricassee.\nRecipe...
5,3622,0.061942,0,american,60,high,"[hamburger meat, cabbage, onion, garlic, salt,...",1041.7,high,"[boil, bake, simmer]",24,high,high,advanced,25.0,high,45110,85,\nThe recipe name:Cabbage Burgers.\nRecipe ins...
6,34582,0.045807,0,american,20,medium,"[potato, beef, onion, water, green bean, salt,...",298.9,medium,"[bake, simmer]",7,medium,medium,intermediate,20.0,high,14081,40,\nThe recipe name:Clean Plate Hamburger Pie.\n...
7,18219,0.045668,0,american,15,high,"[beef, salt, blue cheese, cream cheese, dijon ...",515.8,low,"[grill, sautee]",18,low,high,intermediate,15.0,high,14099,30,\nThe recipe name:Burgers Stuffed with Blue Ch...
8,13064,0.042435,1,american,20,high,"[hamburger meat, salt, sour cream, onion, nonf...",332.3,low,"[boil, simmer, stir_fry]",15,low,medium,intermediate,20.0,high,24502,40,\nThe recipe name:Hamburger Stroganoff.\nRecip...
9,38612,0.038615,1,american,5,high,"[beef, olive oil, garlic, salt, worcestershire...",317.7,low,"[grill, roast]",8,low,high,intermediate,15.0,high,7303,20,\nThe recipe name:Hamburgers.\nRecipe instruct...


### 4.2.2. Collaboration filtering
<a id='section-4.2.2'></a>

For this section, we are using ALS model, which relies on user-item rating interaction and item-item interaction to create the recommendations. A drawback of this method is that without any initial ratings for new user (cold items), we can't make a recommendation for them. Therefore, to address this problem. We are going to have user input some initial preferences or try to figure out their short-term preference from their filter choices.

In [22]:
test_user_profile = USER_PROFILE_TEMPLATE.format(*preference_df.iloc[3][PREFERENCE_COLS].tolist())

print(test_user_profile)


Favorite ingredients are: ['butter', 'egg', 'flour', 'salt', 'sugar'].
Favorite cuisine are: ['american'].
Preferred cooking method: ['bake', 'simmer'].
Preferred cooking difficulty: ['intermediate'].
Preferred Protein content is ['low'].
Preferred Fiber content is ['low'].
Preferred Fat content is ['high'].
Preferred Carbohydrate content is ['high'].
Preferred Sodium content is ['high'].



We try to retrieve the top user profiles that similar to the user

In [23]:
user_retriever = user_vectorstore.as_retriever(
    search_kwargs = dict(
        k=10,  # We retrieve the 10 best similar users
    )
)

retrieved_users =  user_retriever.invoke(test_user_profile)  # retrieve the top search result from the vectorstore that match the query and metadata filter conditions

match_user_ids = np.array([item.id for item in retrieved_users],dtype=int)

retrieved_users[:5]

[Document(id='3', metadata={}, page_content="\nFavorite ingredients are: ['butter', 'egg', 'flour', 'salt', 'sugar'].\nFavorite cuisine are: ['american'].\nPreferred cooking method: ['bake', 'simmer'].\nPreferred cooking difficulty: ['intermediate'].\nPreferred Protein content is ['low'].\nPreferred Fiber content is ['low'].\nPreferred Fat content is ['high'].\nPreferred Carbohydrate content is ['high'].\nPreferred Sodium content is ['high'].\n"),
 Document(id='280', metadata={}, page_content="\nFavorite ingredients are: ['butter', 'egg', 'flour', 'salt', 'sugar'].\nFavorite cuisine are: ['american'].\nPreferred cooking method: ['bake', 'simmer'].\nPreferred cooking difficulty: ['intermediate'].\nPreferred Protein content is ['low'].\nPreferred Fiber content is ['low'].\nPreferred Fat content is ['high'].\nPreferred Carbohydrate content is ['high'].\nPreferred Sodium content is ['high'].\n"),
 Document(id='543', metadata={}, page_content="\nFavorite ingredients are: ['butter', 'egg', '

We find the top recommendations for the top users that are similar to the test user

In [24]:
model_recommendations = als_model.recommend(
    match_user_ids,
    dataset,
    k=k,
    filter_viewed=False
)
model_recommendations.head(3)

,user_id,item_id,score,rank
0,3,57337,17.732025,1
1,3,20444,16.207056,2
2,3,60671,15.921755,3


Finally we us a weighted random selection of recommendations from the pool of recommendations for similar users

In [25]:
# The strategy is to count how many time a recommendation has been suggested for each user, then we have a weighted shuffle
# to select the k recommendations

counts = model_recommendations.item_id.value_counts()

# Exact IDs and the weights
items = counts.index.values
weights = counts.values

# Normalize the weights
probabilities = weights / weights.sum()

np.random.seed(0)  # can turn seed on/off

recommend_ids = np.random.choice(
    items, 
    size=k, 
    replace=False, 
    p=probabilities
)

recommend_ids

array([60671, 39989, 60543, 43328, 50869, 20572, 54913, 25207, 12988,
       45224], dtype=int64)

### 4.2.3. Hybrid filtering
<a id='section-4.2.3'></a>

With normal collaboration models, many of them won't be able to handle cold-start for items without ratings or user without any reviews. Hybrid models are developed to address this problem, one of them is LightFM.

In [26]:
# we are using the same test query
print(test_query)
# and candidate retrieved from vectorstore
candidate_ids[:5]

a meat hamburger


['5114', '29388', '5126', '34582', '60915']

Using the list of candidate, we input that to the lightFM model to rank the top 100 recommendations for each similar users. Note that because the user preference has been taken into context, the top recommendations in the step might have derived from the text_query

In [27]:
recommendations = lightfm_model.recommend(
    users=match_user_ids,  # we also reuse similar users as we don't have rating for new user yet
    dataset=dataset,
    k=100,
    items_to_recommend=np.array(candidate_ids,dtype=int), # Can contain either hot or warm items
    filter_viewed = False
)

In [28]:
# them we sum the score of each recommended items and sort them
top_100 = recommendations.groupby('item_id')['score'].sum().sort_values(ascending=False).index[:100].to_numpy()
top_100[:5]
top_100_docs = vectorstore.get_by_ids(top_100.astype(str))
top_100_docs[:5]

[Document(id='807', metadata={'difficulty': 'intermediate', 'who_score': 0, 'ingredients': ['green bell pepper', 'beef', 'onion', 'tomato', 'long grain rice', 'water', 'salt', 'worcestershire sauce', 'cheddar cheese'], 'carbohydrate_content': 'high', 'cuisine': 'american', 'prep_time': 15, 'calories': 1389.5, 'fat_content': 'high', 'fiber_content': 'high', 'sodium_content': 'high', 'total_time': 50, 'cook_time': 35.0, 'cooking_method': ['boil', 'bake', 'simmer'], 'fsa_score': 22, 'protein_content': 'medium', 'recipe_id': 807}, page_content="\nThe recipe name:Ground Beef Stuffed Green Bell Peppers With Cheese.\nRecipe instruction:\n['Cut off the tops of green peppers; discard seeds and membranes.', 'Chop enough of the tops to make 1/4 cup, set aside.', 'Cook the whole green peppers, uncovered in boiling water for about 5 minutes; invert to drain well.', 'Sprinkle insides of the peppers lightly with salt.', 'In a skillet cook ground beef, onion and 1/4 cup chopped pepper till meat is bro

Then the final step would be to rerank the list recommendation from previous step using the text_query context

In [ ]:
# Need to remapping the document ID because using FlashrankRerank the document id in the result will be overwriten with index from 0->n
doc_ids_map = dict(
    zip(range(len(top_100)),top_100.tolist())
)

reranker = Ranker(
    model_name="ms-marco-MiniLM-L-12-v2",  # NOTE: Can change to a different Flashrank model of your liking
    cache_dir=os.environ["FLASHRANK_PATH"]
)
compressor = FlashrankRerank(client=reranker,top_n=k)

In [30]:
rerank_result = compressor.compress_documents(
    top_100_docs,
    query = test_query #NOTE: can add extra condition here to the query
)

for doc in rerank_result:  # reranking override the doc id so need to change it bank
    doc.metadata['id'] = doc_ids_map[doc.metadata['id']]

data = []

for doc in rerank_result:   # your list of Document objects
    row = doc.metadata.copy()
    row["page_content"] = doc.page_content
    data.append(row)

pd.DataFrame(data)


,id,relevance_score,recipe_id,total_time,ingredients,fsa_score,protein_content,who_score,difficulty,fiber_content,cuisine,fat_content,sodium_content,prep_time,carbohydrate_content,cooking_method,calories,cook_time,page_content
0,24405,0.888597,5126,45,"[beef, onion, salt, catsup, chili sauce, brown...",25,medium,0,intermediate,low,american,high,high,20,low,"[grill, boil, simmer, bake]",824.9,25.0,\nThe recipe name:Barbecue Hamburger Patties.\...
1,41517,0.850926,45110,85,"[hamburger meat, cabbage, onion, garlic, salt,...",24,high,0,advanced,high,american,high,high,60,high,"[boil, bake, simmer]",1041.7,25.0,\nThe recipe name:Cabbage Burgers.\nRecipe ins...
2,20414,0.820912,34582,27,"[hamburger meat, horseradish, worcestershire s...",22,high,0,intermediate,medium,american,high,high,15,high,"[grill, fry, broil]",986.8,12.0,\nThe recipe name:Some Like It Hot Hamburger (...
3,20395,0.814737,3741,60,"[beef, celery, onion, tomato paste, water, ame...",23,medium,0,intermediate,medium,american,high,high,30,medium,"[fry, boil, stew]",1127.7,30.0,\nThe recipe name:Hacienda Hamburger Skillet.\...
4,21676,0.652198,23118,20,"[beef, turkey, butter, green bell pepper, onio...",24,high,0,intermediate,medium,american,high,high,5,high,"[grill, fry, sautee, boil]",852.7,15.0,\nThe recipe name:Mom's Beer Burgers.\nRecipe ...
5,24289,0.599239,5114,25,"[beef, onion, water, evaporated milk, salt, wo...",23,medium,0,intermediate,medium,american,high,high,10,medium,"[broil, grill]",1708.2,15.0,\nThe recipe name:Basic Hamburgers.\nRecipe in...
6,52033,0.526427,52209,110,"[beef, salt, pepper, oregano, basil, seasoning...",14,high,0,advanced,medium,american,high,high,20,high,"[boil, simmer, stew]",478.3,90.0,\nThe recipe name:Bestest Hamburger Soup.\nRec...
7,31005,0.398017,29787,50,"[butter, margarine, onion, mushroom, beef, pot...",23,high,0,advanced,low,american,high,high,30,low,"[sautee, broil, bake]",709.3,20.0,\nThe recipe name:Mushroom-Stuffed Hamburger S...
8,28489,0.099300,25796,20,"[bacon, beef, worcestershire sauce, salt, garl...",23,high,0,intermediate,low,american,high,high,10,low,"[fry, sautee, grill]",747.9,10.0,\nThe recipe name:The Peanut Bacon Burger.\nRe...
9,35758,0.097542,34566,75,"[onion, water, pepper, tomato paste, tomato sa...",21,high,0,advanced,medium,american,high,high,45,high,"[boil, simmer, bake, roast]",650.5,30.0,\nThe recipe name:Hungry Jack Topped Hamburger...
